# Optimización Automática de Hiperparámetros con Keras Tuner

En este notebook aprenderás cómo usar **Keras Tuner** para ajustar automáticamente los hiperparámetros de una red neuronal. Utilizaremos el dataset **Iris** para clasificar flores en tres especies.

## Objetivos
- Comprender qué son los hiperparámetros.
- Aplicar Keras Tuner para optimizar redes neuronales.
- Seleccionar la mejor cantidad de neuronas, tasa de aprendizaje y batch size.
- Comparar el rendimiento del mejor modelo encontrado.

## Los hiperparámetros

Los hiperparámetros son configuraciones que se establecen antes de que comience el proceso de entrenamiento de un modelo de machine learning o red neuronal. A diferencia de los parámetros del modelo (como los pesos y sesgos de una red neuronal), que se aprenden automáticamente a partir de los datos durante el entrenamiento, los hiperparámetros no se aprenden.

Piensa en ellos como los "ajustes" de alto nivel que controlan cómo se entrena un modelo y la estructura del modelo mismo. Ejemplos comunes de hiperparámetros incluyen:

- Tasa de aprendizaje (Learning Rate): Qué tan grandes son los pasos que da el optimizador al actualizar los pesos del modelo.
- Número de épocas (Number of Epochs): Cuántas veces completas el conjunto de datos de entrenamiento se pasa a través de la red.
- Tamaño del lote (Batch Size): El número de muestras de entrenamiento utilizadas en una sola iteración del entrenamiento.
- Número de capas ocultas (Number of Hidden Layers): Cuántas capas intermedias tiene una red neuronal.
- Número de neuronas por capa (Number of Neurons per Layer): Cuántas unidades hay en cada capa oculta.
- Función de activación (Activation Function): La función utilizada en las neuronas para introducir no linealidad.
- Regularización (Regularization): Técnicas para evitar el sobreajuste, como L1 o L2.

La elección correcta de los hiperparámetros es crucial porque puede tener un impacto significativo en el rendimiento del modelo. Encontrar la combinación óptima de hiperparámetros suele ser un proceso experimental, y herramientas como Keras Tuner ayudan a automatizar esta búsqueda.

En este notebook, por ejemplo, estamos optimizando el número de neuronas por capa (units), el número de capas ocultas (num_layers) y la tasa de aprendizaje (learning_rate) para encontrar la mejor combinación para clasificar las flores Iris.

## 1. Cargar y preparar los datos

In [ ]:
# Instalación manual si falta una dependencia: %pip install keras-tuner -q

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

In [ ]:
# Cargar el dataset Iris
iris = load_iris()
X = iris.data
y = iris.target

In [ ]:
# El escalador se ajusta después de separar entrenamiento, validación y prueba.

In [ ]:
y

In [ ]:
# Convertir etiquetas a one-hot encoding
y_cat = to_categorical(y)

In [ ]:
import numpy as np
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=np.argmax(y_train, axis=1))
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
print('Train / validación / test:', len(X_train), len(X_val), len(X_test))


## 2. Definir el modelo y los hiperparámetros a optimizar

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
from tensorflow import keras
from keras import layers
import keras_tuner as kt

def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))

    # Número de capas ocultas
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(layers.Dense(
            units=hp.Int(f'units_{i}', min_value=8, max_value=64, step=8),
            activation='relu'
        ))

    model.add(layers.Dense(3, activation='softmax'))

    learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

## 3. Configurar el *Tuner* y ejecutar la búsqueda de hiperparámetros

- build_model: Esta es la función que define tu modelo Keras. Keras Tuner usará esta función para construir diferentes arquitecturas y configuraciones de modelo basadas en los hiperparámetros que explore.

- objective='val_accuracy': Esto especifica la métrica que Keras Tuner intentará optimizar durante la búsqueda de hiperparámetros. En este caso, el objetivo es maximizar la precisión de validación (val_accuracy).

- max_epochs=50: Este es el número máximo de épocas para entrenar un modelo durante la búsqueda de hiperparámetros. Hyperband es un método basado en parada temprana, por lo que los modelos podrían detenerse antes de max_epochs si no están funcionando bien.

- factor=3: Este es el factor de reducción para el número de modelos y épocas en cada "bracket" (ronda) del algoritmo Hyperband. Un factor de 3 significa que en cada bracket sucesivo, el número de modelos se reduce por un factor de 3, y los modelos restantes se entrenan por 3 veces más épocas.

- directory='autotuner_logs': Este es el directorio donde Keras Tuner almacenará los registros y resultados de la búsqueda de hiperparámetros. Esto incluye información sobre los diferentes intentos, los hiperparámetros explorados y el rendimiento de cada modelo.

- project_name='iris_tuning': Este es el nombre del proyecto. Se utiliza para organizar los resultados dentro del directorio especificado.

In [ ]:
from datetime import datetime
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=50,
    factor=3,
    directory='autotuner_logs',
    seed=42, project_name='iris_tuning_' + run_id
)

stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
tuner.search(X_train, y_train, epochs=50, validation_data=(X_val, y_val), callbacks=[stop_early])

## 4. Evaluar el mejor modelo encontrado

- tuner.get_best_hyperparameters(): Este es un método del objeto tuner que recupera las mejores configuraciones de hiperparámetros encontradas durante el proceso de búsqueda.

- num_trials=1: Este argumento especifica cuántas de las mejores configuraciones de hiperparámetros deseas recuperar. En este caso, estamos pidiendo solo la mejor configuración (num_trials=1).

- [0]: El método get_best_hyperparameters() devuelve una lista de objetos de hiperparámetros (incluso si solicitas solo uno). El [0] al final se utiliza para acceder al primer elemento de esa lista, que corresponde a la mejor configuración de hiperparámetros.

**Lectura de la salida:** la métrica `accuracy` que la celda imprime como «Precisión final» corresponde a **exactitud** (proporción total de aciertos). No es la métrica `precision` de la clase positiva.


In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print('Mejores hiperparámetros:')
print(f"Capas ocultas: {best_hps.get('num_layers')}")
for i in range(best_hps.get('num_layers')):
    print(f"  - Neuronas capa {i+1}: {best_hps.get(f'units_{i}')}")
print(f"Learning rate: {best_hps.get('learning_rate')}")

In [ ]:
best_model = tuner.hypermodel.build(best_hps)
# validation_data=(X_val, y_val):
# Esto especifica que el 20% de los datos de entrenamiento
# se reservarán como conjunto de validación durante el entrenamiento.
# Se monitorea el rendimiento del modelo en este conjunto de validación.
history = best_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), verbose=0)
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f'\nPrecisión final en test: {test_acc:.4f}')

## 5. Visualización del entrenamiento

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Evolución de la precisión durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.show()

## Conclusiones
- **Keras Tuner** permite explorar automáticamente configuraciones óptimas sin intervención manual.
- Se optimizaron el número de capas, neuronas y la tasa de aprendizaje.
- La automatización acelera el desarrollo de modelos eficientes.
- Esta técnica es útil especialmente cuando el espacio de búsqueda es grande o el modelo es sensible a los hiperparámetros.